In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

# Policy network
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.fc(state)

# Environment and setup
env = gym.make("CartPole-v1")
policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
optimizer = optim.Adam(policy.parameters(), lr=1e-2)
gamma = 0.99

for episode in range(500):
    states, actions, rewards = [], [], []
    state = env.reset()[0]
    done = False

    # 1️⃣ Run an episode
    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32)
        probs = policy(state_tensor)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()

        next_state, reward, done, _, _ = env.step(action.item())
        states.append(state_tensor)
        actions.append(action)
        rewards.append(reward)
        state = next_state

    # 2️⃣ Compute returns (discounted rewards)
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # normalize

    # 3️⃣ Compute policy loss
    log_probs = []
    for state, action in zip(states, actions):
        probs = policy(state)
        dist = torch.distributions.Categorical(probs)
        log_probs.append(dist.log_prob(action))
    log_probs = torch.stack(log_probs)

    loss = -(log_probs * returns).sum()

    # 4️⃣ Backpropagate
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Episode {episode}, total reward: {sum(rewards):.2f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

# --- Policy Network ---
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.net(state)

# --- REINFORCE Agent ---
def reinforce(env_name="CartPole-v1", lr=1e-3, gamma=0.99, episodes=500):
    env = gym.make(env_name)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    for ep in range(episodes):
        log_probs, rewards = [], []
        state = env.reset()[0] if isinstance(env.reset(), tuple) else env.reset()

        done = False
        while not done:
            state_t = torch.tensor(state, dtype=torch.float32)
            probs = policy(state_t)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))

            next_state, reward, done, *_ = env.step(action.item())
            rewards.append(reward)
            state = next_state

        # Compute discounted returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # Compute policy gradient loss
        loss = -torch.sum(torch.stack(log_probs) * returns)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Episode {ep+1}: total reward = {sum(rewards)}")

    env.close()

reinforce()


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

def scrape_interesting_engineering(max_pages=100):
    # Chrome options
    options = Options()
    options.add_argument("--headless")  # comment this line if you want to see the browser
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    # Start driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get("https://interestingengineering.com/news")

    articles = []
    page = 0

    while page < max_pages:
        page += 1
        print(f"📰 Scraping page {page}...")

        # Wait for articles to load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "article"))
            )
        except Exception:
            print("❌ No articles found, stopping.")
            break

        # Parse HTML
        soup = BeautifulSoup(driver.page_source, "html.parser")
        found = 0
        for article in soup.select("article"):
            title_tag = article.find("a")
            date_tag = article.find("time")
            if title_tag:
                title = title_tag.get_text(strip=True)
                date = date_tag.get_text(strip=True) if date_tag else "No date"
                articles.append((title, date))
                found += 1
        print(f"  Found {found} articles on this page.")

        # Try to click NEXT
        try:
            next_btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, "//a[contains(translate(., 'next', 'NEXT'), 'NEXT')]"))
            )
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(2)  # wait for next page to load
        except Exception:
            print("✅ Reached last page.")
            break

    driver.quit()
    return articles


def save_to_text(articles, filename="InterestingEngineering_News.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        for title, date in articles:
            f.write(f"{title} — {date}\n")
    print(f"✅ Saved {len(articles)} articles to {filename}")


if __name__ == "__main__":
    data = scrape_interesting_engineering()
    save_to_text(data)


📰 Scraping page 1...
  Found 1 articles on this page.
📰 Scraping page 2...
  Found 1 articles on this page.
📰 Scraping page 3...
  Found 10 articles on this page.
✅ Reached last page.


RemoteDisconnected: Remote end closed connection without response

In [ ]:
You know ResNet and how skipping happens in every block. In ResNet models, skipping happens only for individual blocks. 
I imagine a model where skipping happens for any group of contiguous blocks. 
This will improve smooth specialization between blocks. 
Please code my idea.

In [ ]:
Nice idea — contiguous-group skipping should let neighboring blocks specialize smoothly. 
Below is a ready-to-run PyTorch implementation of a Group-Skip ResNet that implements learned contiguous window gating over successive residual blocks.

In [ ]:
Key idea

A small gating network (an MLP from a global pooled feature) predicts, per sample, a window center c and half-width w.
For block index i we form a differentiable mask
mask_i = sigmoid(k * (w - |i - c|))
so blocks within the window (|i-c| < w) get mask ≈ 1, blocks outside get mask ≈ 0. k controls sharpness.
Each block's output is scaled by its mask before adding into the residual path. When mask ≈ 0 → block is effectively skipped.
At inference you can make masks hard (threshold) if desired.

In [ ]:
The code below provides:
BasicBlock — a simple ResNet-like block
GroupSkipResNet — stacks blocks and applies the group-skip gating
options for temperature k, L1 regularization on width to encourage narrow windows, and hard thresholding at inference

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional

class BasicBlock(nn.Module):
    """A simplified ResNet basic block (no downsample handling here for brevity).
       Assumes input and output channels are equal and stride=1."""
    def __init__(self, channels: int, norm_layer=nn.BatchNorm2d):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = norm_layer(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = norm_layer(channels)

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        # standard residual add happens outside (we want to apply mask to out)
        return out


class GroupSkipResNet(nn.Module):
    """ResNet-like backbone with learnable contiguous-group skipping.
       - num_blocks: total number of successive basic blocks
       - channels: channel count kept constant for simplicity
       - k: sharpness of sigmoid window (larger -> closer to hard window)
       - gate_hidden: hidden dim for gating MLP"""
    def __init__(self,
                 num_blocks: int,
                 channels: int = 64,
                 k: float = 8.0,
                 gate_hidden: int = 64,
                 norm_layer=nn.BatchNorm2d):
        super().__init__()
        assert num_blocks >= 1
        self.num_blocks = num_blocks
        self.channels = channels
        self.k = float(k)

        # initial conv / stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, kernel_size=7, stride=2, padding=3, bias=False),
            norm_layer(channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # stack of basic blocks (same channels here for simplicity)
        self.blocks = nn.ModuleList([BasicBlock(channels, norm_layer=norm_layer) for _ in range(num_blocks)])

        # gating MLP: map global pooled features -> center (c) and half-width (w)
        # we predict in a normalized coordinate system where block indices run 0..num_blocks-1
        self.gate_mlp = nn.Sequential(
            nn.Linear(channels, gate_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(gate_hidden, 2)   # outputs: raw_c, raw_w
        )

        # small learnable scale and bias to make initial gating sensible
        # initialize bias such that c ~ (num_blocks-1)/2 and w ~ num_blocks/2 (so all blocks active initially)
        self.register_buffer("block_indices", torch.arange(num_blocks, dtype=torch.float32).unsqueeze(0))  # shape (1, N)
        # We'll use a tiny offset initializer (bias in gate_mlp second layer will handle initial range)

    def forward(self, x, hard_threshold: Optional[float] = None):
        """
        x: input images (B,3,H,W)
        hard_threshold: if provided, threshold masks at this value for hard gating (inference)
        Returns:
          out: tensor after passing through blocks with group-skip
          masks: (B, num_blocks) float masks per sample (soft in training, may be hard if threshold provided)
        """
        B = x.shape[0]
        out = self.stem(x)                 # (B, C, H', W')

        # Global pooled feature for gating (per-sample)
        gp = F.adaptive_avg_pool2d(out, (1, 1)).view(B, -1)  # (B, channels)

        # Predict raw center and width
        raw = self.gate_mlp(gp)           # (B, 2)
        raw_c = raw[:, 0]                 # center
        raw_w = raw[:, 1]                 # half-width (can be negative initially)

        # Map raw outputs to meaningful ranges:
        # center c in [0, num_blocks-1], width w in [0, num_blocks]
        c = (torch.sigmoid(raw_c) * (self.num_blocks - 1)).unsqueeze(1)   # (B,1)
        w = (torch.sigmoid(raw_w) * self.num_blocks).unsqueeze(1)         # (B,1)

        # compute differentiable window masks per block index
        # block_indices shape (1, N), broadcast with (B,1) -> (B,N)
        indices = self.block_indices.to(x.device)  # (1, N)
        dist = torch.abs(indices - c)              # (B, N)
        # mask = sigmoid(k * (w - dist))
        masks = torch.sigmoid(self.k * (w - dist))  # (B, N) in (0,1)

        # Optionally hard-threshold at inference
        if hard_threshold is not None:
            masks = (masks >= float(hard_threshold)).float()

        # sequentially apply blocks with per-sample mask
        # note: mask for block i is shape (B,) -> we reshape to (B,1,1) to broadcast to (B,C,H,W)
        for i, block in enumerate(self.blocks):
            block_out = block(out)  # (B, C, H, W)
            mi = masks[:, i].view(B, 1, 1, 1)  # broadcast
            out = out + block_out * mi

        return out, masks

    def get_mask_regularizer(self, masks):
        """Return a regularizer encouraging compact windows.
           For example, encourage small widths by L1 on masks sum (smaller sum -> narrower).
           masks: (B, N)"""
        # L1 on per-sample mask sums approximates window width
        width_est = masks.sum(dim=1)          # (B,)
        return width_est.mean()               # scalar

# Example usage
if __name__ == "__main__":
    model = GroupSkipResNet(num_blocks=8, channels=32, k=10.0)
    images = torch.randn(4, 3, 224, 224)
    out, masks = model(images)
    print("out shape:", out.shape)         # expected (B, channels, H', W')
    print("masks:", masks)                 # per-sample, per-block masks


out shape: torch.Size([4, 32, 56, 56])
masks: tensor([[0.4084, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9763],
        [0.3933, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9755],
        [0.4038, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9789],
        [0.4029, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9743]],
       grad_fn=<SigmoidBackward0>)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import h5py
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from s2cnn import S2Convolution, S2Activation

# ------------------------------
# 1️⃣ Spherical CNN Model
# ------------------------------
class SphericalWeatherNet(nn.Module):
    def __init__(self, in_channels=12, out_channels=1, bandwidth=64):
        super().__init__()
        self.conv1 = S2Convolution(in_channels, 32, bandwidth, bandwidth//2)
        self.act1 = S2Activation(nn.ReLU())
        self.conv2 = S2Convolution(32, 64, bandwidth//2, bandwidth//4)
        self.act2 = S2Activation(nn.ReLU())
        self.conv3 = S2Convolution(64, out_channels, bandwidth//4, bandwidth//8)

    def forward(self, x):
        x = self.act1(self.conv1(x))
        x = self.act2(self.conv2(x))
        x = self.conv3(x)
        return x

# ------------------------------
# 2️⃣ Grid + mask helper
# ------------------------------
def make_era5_to_s2cnn_grid_lat_range(lat, lon, bandwidth=64, lat_min=0, lat_max=90):
    b = bandwidth
    beta = np.linspace(0, np.pi, 2*b, endpoint=False)
    alpha = np.linspace(0, 2*np.pi, 2*b, endpoint=False)
    beta_grid, alpha_grid = np.meshgrid(beta, alpha, indexing="ij")
    lat_sph = 90 - np.degrees(beta_grid)
    lon_sph = np.degrees(alpha_grid)
    lon_norm = (lon_sph / 180.0) - 1.0
    lat_norm = -lat_sph / 90.0
    grid = np.stack([lon_norm, lat_norm], axis=-1)
    grid = torch.from_numpy(grid).unsqueeze(0).float().cuda()
    mask = ((lat_sph >= lat_min) & (lat_sph <= lat_max)).astype(np.float32)
    mask = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0).cuda()
    return grid, mask

# ------------------------------
# 3️⃣ ERA5 -> S2CNN conversion
# ------------------------------
def era5_to_s2cnn_batch(X, grid_s2cnn, mask=None):
    is_sequence = (X.dim() == 4)
    if is_sequence:
        T, C, H, W = X.shape
        X_flat = X.reshape(T*C, 1, H, W)
    else:
        C, H, W = X.shape
        X_flat = X.unsqueeze(0)
    X_s2cnn = F.grid_sample(X_flat, grid_s2cnn, align_corners=True)
    if mask is not None:
        X_s2cnn *= mask
    if is_sequence:
        X_s2cnn = X_s2cnn.reshape(T, C, X_s2cnn.shape[-2], X_s2cnn.shape[-1])
    else:
        X_s2cnn = X_s2cnn.squeeze(0)
    return X_s2cnn

# ------------------------------
# 4️⃣ S2CNN -> ERA5 conversion
# ------------------------------
def s2cnn_to_era5_batch(Y_s2cnn, lat, lon):
    is_sequence = (Y_s2cnn.dim() == 5)
    if is_sequence:
        B, T, C, Hs, Ws = Y_s2cnn.shape
        Y_flat = Y_s2cnn.reshape(B*T, C, Hs, Ws)
    else:
        B, C, Hs, Ws = Y_s2cnn.shape
        Y_flat = Y_s2cnn
    H, W = len(lat), len(lon)
    lon_grid, lat_grid = np.meshgrid(lon, lat)
    lon_norm = (lon_grid / 180.0) - 1.0
    lat_norm = -lat_grid / 90.0
    grid = np.stack([lon_norm, lat_norm], axis=-1)
    grid = torch.from_numpy(grid).unsqueeze(0).float().cuda()
    Y_era5 = F.grid_sample(Y_flat, grid, align_corners=True)
    if is_sequence:
        Y_era5 = Y_era5.reshape(B, T, C, H, W)
    return Y_era5

# ------------------------------
# 5️⃣ Dataset (streaming version)
# ------------------------------
class ERA5StreamingDataset(Dataset):
    def __init__(self, file_list, T_in=3, lat_crop=(0,90), bandwidth=64):
        self.file_list = file_list
        self.T_in = T_in
        self.lat_min, self.lat_max = lat_crop
        self.bandwidth = bandwidth

        # Load grid from first file
        with h5py.File(file_list[0], 'r') as f:
            self.lat = f['lat'][:]
            self.lon = f['lon'][:]

        # Precompute S2CNN grid & mask
        self.grid_s2cnn, self.mask = make_era5_to_s2cnn_grid_lat_range(
            self.lat, self.lon, bandwidth=self.bandwidth,
            lat_min=self.lat_min, lat_max=self.lat_max
        )
        self.grid_s2cnn = self.grid_s2cnn.cuda()
        self.mask = self.mask.cuda()

        # Compute sample offsets
        self.sample_offsets = []
        offset = 0
        for fname in file_list:
            with h5py.File(fname,'r') as f:
                N = f['X'].shape[0]
                self.sample_offsets.append((offset, offset+N))
                offset += N
        self.total_samples = offset

    def __len__(self):
        return self.total_samples - self.T_in

    def __getitem__(self, idx):
        for file_idx, (start,end) in enumerate(self.sample_offsets):
            if start <= idx < end - self.T_in:
                idx_in_file = idx - start
                file_name = self.file_list[file_idx]
                break
        else:
            raise IndexError(f"{idx} out of range")

        with h5py.File(file_name,'r') as f:
            X_seq = f['X'][idx_in_file:idx_in_file+self.T_in]
            Y = f['Y'][idx_in_file+self.T_in]

        X_seq = torch.from_numpy(X_seq).float()
        Y = torch.from_numpy(Y).float()
        X_s2cnn = era5_to_s2cnn_batch(X_seq, self.grid_s2cnn, self.mask)
        return X_s2cnn, Y

# ------------------------------
# 6️⃣ Recursive multi-day forecast
# ------------------------------
def recursive_forecast(model, X_init, steps, grid_s2cnn, mask=None, lat=None, lon=None):
    B, T_in, C_in, H, W = X_init.shape
    X_seq = X_init.clone().cuda()
    forecasts = []

    for t in range(steps):
        X_s2cnn = era5_to_s2cnn_batch(X_seq, grid_s2cnn, mask)
        X_flat = X_s2cnn.reshape(B, T_in*C_in, X_s2cnn.shape[-2], X_s2cnn.shape[-1])
        with torch.no_grad():
            y_pred_s2cnn = model(X_flat)
            y_pred_era5 = s2cnn_to_era5_batch(y_pred_s2cnn, lat, lon)
        forecasts.append(y_pred_era5.cpu())

        # Update input sequence
        if T_in > 1:
            X_seq = torch.cat([X_seq[:,1:,:,:,:], y_pred_era5.unsqueeze(1)], dim=1)
        else:
            X_seq = y_pred_era5.unsqueeze(1)

    forecasts = torch.stack(forecasts, dim=1)
    return forecasts

# ------------------------------
# 7️⃣ Visualization
# ------------------------------
def plot_forecasts_global(forecasts, lat, lon, step_indices=None, variable_index=0, cmap='coolwarm'):
    if isinstance(forecasts, torch.Tensor):
        forecasts = forecasts.cpu().numpy()
    B, steps, C_out, H, W = forecasts.shape
    if step_indices is None:
        step_indices = range(steps)
    for b in range(B):
        for t in step_indices:
            data = forecasts[b,t,variable_index]
            plt.figure(figsize=(12,6))
            ax = plt.axes(projection=ccrs.PlateCarree())
            ax.coastlines()
            mesh = ax.pcolormesh(lon, lat, data, cmap=cmap, shading='auto')
            plt.title(f"Sample {b}, Forecast step {t}")
            plt.colorbar(mesh, label='Value')
            plt.show()

# ------------------------------
# 8️⃣ Metrics computation
# ------------------------------
def compute_forecast_metrics(forecasts, y_true, step_indices=None, variable_index=0):
    if isinstance(forecasts, torch.Tensor):
        forecasts = forecasts.cpu().numpy()
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.cpu().numpy()
    B, steps, C_out, H, W = forecasts.shape
    if step_indices is None:
        step_indices = range(steps)
    metrics = {}
    for t in step_indices:
        pred = forecasts[:, t, variable_index]
        true = y_true[:, t, variable_index]
        error = pred - true
        rmse = np.sqrt(np.mean(error**2))
        mae = np.mean(np.abs(error))
        mean_error = np.mean(error)
        metrics[t] = {"RMSE":rmse,"MAE":mae,"MeanError":mean_error}
    return metrics

# ------------------------------
# 9️⃣ Example: Training loop
# ------------------------------
# Example: random data for demonstration
H, W = 721, 1440
B, T_in, C_in, C_out = 2, 3, 4, 1
X_train = torch.randn(B,T_in,C_in,H,W).cuda()
Y_train = torch.randn(B,C_out,H,W).cuda()
lat = np.linspace(90,-90,H)
lon = np.linspace(0,359.75,W)
bandwidth=64
grid_s2cnn, mask_north = make_era5_to_s2cnn_grid_lat_range(lat, lon, bandwidth, 0, 90)
grid_s2cnn = grid_s2cnn.cuda()
mask_north = mask_north.cuda()

dataset = ERA5StreamingDataset(file_list=[], T_in=T_in, lat_crop=(0,90), bandwidth=bandwidth)
# Overwrite with random data
dataset.X = X_train
dataset.Y = Y_train
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

model = SphericalWeatherNet(in_channels=T_in*C_in, out_channels=C_out, bandwidth=bandwidth).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(2):
    model.train()
    for x_batch, y_batch in dataloader:
        B, T, C, Hs, Ws = x_batch.shape
        x_flat = x_batch.reshape(B,T*C,Hs,Ws).cuda()
        y_batch = y_batch.cuda()
        optimizer.zero_grad()
        y_pred_s2cnn = model(x_flat)
        y_pred_era5 = s2cnn_to_era5_batch(y_pred_s2cnn, lat, lon)
        loss = criterion(y_pred_era5, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.6f}")

# ------------------------------
# 10️⃣ Recursive forecast + plot + metrics
# ------------------------------
X_init = X_train.clone()
forecast_steps = 3
forecasts = recursive_forecast(model, X_init, forecast_steps, grid_s2cnn, mask_north, lat, lon)
plot_forecasts_global(forecasts, lat, lon, step_indices=[0,1,2])
metrics = compute_forecast_metrics(forecasts, Y_train.unsqueeze(1).repeat(1,forecast_steps,1,1,1))
print(metrics)
